In [ ]:
%load_ext autoreload
%autoreload 2

# 02 — EDA on the real Findex data

Who are the 18,718 applicants? Survey data only, nothing simulated. All shares are weighted within each country; the "20-country average" gives every country equal weight.

In [1]:
import pandas as pd

from mkopoguard import config
from mkopoguard.stats import pool_average, weighted_share
from mkopoguard.viz import apply_style, paired_bars, ranked_bars, save

apply_style()
IMG = config.PROJECT_ROOT / "docs" / "img"
C = ["country_code"]

applicants = pd.read_parquet(config.DATA_INTERIM / "findex_clean.parquet")
names = (
    applicants.drop_duplicates("country_code")
    .set_index("country_code")["country"]
    .astype(str)
    .to_dict()
)
names["CIV"] = "Côte d'Ivoire"  # the raw file stores this name with a broken character
print(f"{len(applicants):,} applicants")

18,718 applicants


In [2]:
mm = weighted_share(applicants, "has_mobile_money", C)
fig = ranked_bars(
    mm,
    "Kenya leads the pool in mobile money",
    "Adults (18+) with a mobile money account, weighted, Findex 2021-22",
    labels=names,
)
save(fig, IMG / "eda_01_mobile_money_by_country.png")
mm.sort_values(ascending=False).round(3)

country_code
KEN    0.708
GHA    0.617
SWZ    0.617
GAB    0.593
UGA    0.554
ZWE    0.532
LSO    0.467
SEN    0.465
TZA    0.463
NAM    0.449
CMR    0.441
ZMB    0.439
CIV    0.429
BEN    0.389
COG    0.382
TGO    0.379
BWA    0.377
ZAF    0.375
LBR    0.374
MOZ    0.291
dtype: float64

**Takeaway:** Kenya has the highest mobile money ownership in the pool (71%); Mozambique the lowest (29%). Weighted and adults-only, seven pool countries fall just under the 40% cut-off used to select them (that rule was unweighted and included 15–17 year olds).

In [3]:
rows = {}
for col, label in [("has_bank_account", "Bank account"), ("has_mobile_money", "Mobile money")]:
    rows[label] = pool_average(weighted_share(applicants, col, C + ["is_female"]))
table = pd.DataFrame(rows).T.rename(columns={0: "Men", 1: "Women"})[["Men", "Women"]]
fig = paired_bars(
    table,
    "Women are less likely to hold either kind of account",
    "Adults (18+), weighted, average of 20 countries",
)
save(fig, IMG / "eda_02_accounts_by_gender.png")
table.round(3)

is_female,Men,Women
Bank account,0.426,0.324
Mobile money,0.508,0.429


**Takeaway:** women trail men by about 10 points for bank accounts and 8 for mobile money. Fewer women will have a statement, so the model sees less about them: a fairness risk to audit in Phase 4.

In [4]:
s = weighted_share(applicants, "has_mobile_money", C + ["income_quintile"])
table = pd.DataFrame({"Kenya": s.loc["KEN"], "20-country average": pool_average(s)})
fig = paired_bars(
    table,
    "Mobile money rises steadily with income",
    "Adults with a mobile money account by household income quintile (1 = poorest 20%)",
    xlabel="Income quintile",
)
save(fig, IMG / "eda_03_mobile_money_by_income.png")
table.round(3)

,Kenya,20-country average
income_quintile,,
1,0.513,0.314
2,0.658,0.398
3,0.734,0.468
4,0.779,0.518
5,0.833,0.628


**Takeaway:** ownership doubles from the poorest to the richest quintile across the pool (31% → 63%). Even Kenya's poorest quintile (51%) is above the pool average. Thin-file applicants will skew poorer.

In [5]:
rows = {}
for col, label in [
    ("borrowed_past_year", "Borrowed at all"),
    ("borrowed_from_family_friends", "From family or friends"),
    ("borrowed_from_fin_institution", "From a financial institution"),
    ("saved_in_savings_club", "Saved in a savings club"),
]:
    s = weighted_share(applicants, col, C)
    rows[label] = {"Kenya": s.loc["KEN"], "20-country average": pool_average(s).iloc[0]}
table = pd.DataFrame(rows).T
fig = paired_bars(
    table, "Most borrowing is informal, even in Kenya", "Adults (18+) in the past year, weighted"
)
save(fig, IMG / "eda_04_borrowing_sources.png")
table.round(3)

,Kenya,20-country average
Borrowed at all,0.775,0.573
From family or friends,0.551,0.403
From a financial institution,0.211,0.087
Saved in a savings club,0.326,0.258


**Takeaway:** 77% of Kenyan adults borrowed last year, but only 21% from a financial institution vs 55% from family or friends. Most borrowers have no formal credit history: the gap MkopoGuard targets.

In [6]:
s = weighted_share(applicants, "emergency_fund_source", C + ["income_quintile"], value="could_not")
table = pd.DataFrame({"20-country average": pool_average(s)})
fig = paired_bars(
    table,
    "The poorest are three times as likely to have no emergency fallback",
    "Adults who could not raise emergency funds within 30 days, by income quintile",
    xlabel="Income quintile",
)
save(fig, IMG / "eda_05_no_emergency_funds_by_income.png")
table.round(3)

,20-country average
income_quintile,
1,0.202
2,0.157
3,0.118
4,0.099
5,0.064


**Takeaway:** 20% of the poorest quintile could not raise emergency money in 30 days vs 6% of the richest. This answer feeds the simulator's stress trait. Kenya is not shown: about 190 respondents per quintile is too few for a reliable share.